# 04 — Enrich YSU Course Records

Fetches YSU program pages (live HTTP requests) to enrich YSU rows in `final_curriculum_dataset.csv` with:
- `semester` — extracted from accordion panels
- `description` — from section 3 of each course detail
- `assessment` — Exam / Attestation / Defense

**Input/Output:** `data/processed/university/final_curriculum_dataset.csv` (modified in-place)

> **Note:** This makes ~19 HTTP requests to ysu.am. Run after `03_build_curriculum.ipynb`. Already run — only re-run if dataset is rebuilt.

In [ ]:
import os, re, time
import requests
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup

BASE = Path.cwd().parent.parent
PROC_DIR = BASE / "data" / "processed" / "university"
FINAL = PROC_DIR / "final_curriculum_dataset.csv"

HEADERS = {"User-Agent": "ThesisResearch/1.0 (Armenian IT curriculum alignment; academic use)"}
DELAY_S = 1.2

## Helper functions

In [ ]:
def normalize_url(url):
    """Canonicalize ysu.am/www.ysu.am URL for matching."""
    return url.strip().replace("https://ysu.am/", "https://www.ysu.am/")


def normalize_text(t):
    """Collapse whitespace for matching."""
    return " ".join(str(t).split())


def extract_semester_number(sem_text):
    """Extract integer semester number from Armenian semester text. Returns int or None."""
    m = re.search(r"(\d+)", sem_text)
    return int(m.group(1)) if m else None


def get_field_text(article, css_name):
    """Extract .field__item text from a field div by field--name-* CSS class fragment."""
    div = article.find("div", class_=lambda c: c and css_name in c)
    if not div:
        return ""
    item = div.find(class_="field__item")
    return normalize_text(item.get_text(" ", strip=True)) if item else ""


def classify_assessment(criteria_text):
    """
    Classify assessment type from the evaluation criteria text.
    Armenian keywords built via chr() to avoid source-file encoding issues.
    """
    QNNWTYUN_U      = "".join(chr(c) for c in [0x554,0x576,0x576,0x578,0x582,0x569,0x575,0x578,0x582,0x576])
    QNNWTYUN_L      = "".join(chr(c) for c in [0x584,0x576,0x576,0x578,0x582,0x569,0x575,0x578,0x582,0x576])
    QNNWTYUN_STEM_U = "".join(chr(c) for c in [0x554,0x576,0x576,0x578,0x582,0x569,0x575])
    QNNWTYUN_STEM_L = "".join(chr(c) for c in [0x584,0x576,0x576,0x578,0x582,0x569,0x575])
    STUGARK_U   = "".join(chr(c) for c in [0x54D,0x57F,0x578,0x582,0x563,0x561,0x580,0x584])
    STUGARK_L   = "".join(chr(c) for c in [0x57D,0x57F,0x578,0x582,0x563,0x561,0x580,0x584])
    STUGARK_TAIL= "".join(chr(c) for c in [0x57F,0x578,0x582,0x563,0x561,0x580,0x584])
    STUQMAN_U   = "".join(chr(c) for c in [0x54D,0x57F,0x578,0x582,0x563,0x574,0x561,0x576])
    STUQMAN_L   = "".join(chr(c) for c in [0x57D,0x57F,0x578,0x582,0x563,0x574,0x561,0x576])
    ATESTAVORMAN = "".join(chr(c) for c in [0x561,0x57F,0x565,0x57D,0x57F,0x561,0x57E,0x578,0x580,0x574,0x561,0x576])
    EZRAFAKIC_L = "".join(chr(c) for c in [0x565,0x566,0x580,0x561,0x583,0x561,0x56F,0x56B,0x579])
    EZRAFAKIC_U = "".join(chr(c) for c in [0x535,0x566,0x580,0x561,0x583,0x561,0x56F,0x56B,0x579])
    ENTQACIQ_L  = "".join(chr(c) for c in [0x568,0x576,0x569,0x561,0x581,0x56B,0x56F])

    t = criteria_text
    t_lower = t.lower()

    # --- EXAM ---
    if QNNWTYUN_U in t or QNNWTYUN_L in t:
        return "Exam"
    if QNNWTYUN_STEM_U in t or QNNWTYUN_STEM_L in t:
        return "Exam"
    if "final exam" in t_lower:
        return "Exam"
    if "ends with an oral exam" in t_lower or "oral exam" in t_lower:
        return "Exam"
    if "the exam" in t_lower and "test" not in t_lower:
        return "Exam"

    # --- ATTESTATION ---
    if STUGARK_U in t or STUGARK_L in t:
        return "Attestation"
    if t.startswith(STUGARK_TAIL):
        return "Attestation"
    if "attestation" in t_lower or "credit test" in t_lower:
        return "Attestation"
    if "ends with a test" in t_lower or "in the form of a test" in t_lower:
        return "Attestation"
    if "test is conducted" in t_lower or "conducted by questionnaire" in t_lower:
        return "Attestation"
    if "pass/no pass" in t_lower or "pass or no pass" in t_lower:
        return "Attestation"
    if t_lower.startswith("test.") or t_lower.startswith("test "):
        return "Attestation"
    if "midterm exam" in t_lower and "final exam" not in t_lower:
        return "Attestation"
    if "current exam" in t_lower and "final exam" not in t_lower:
        return "Attestation"
    if ENTQACIQ_L in t and EZRAFAKIC_L not in t and EZRAFAKIC_U not in t:
        if QNNWTYUN_L not in t and QNNWTYUN_U not in t and QNNWTYUN_STEM_L not in t:
            return "Attestation"
    if STUQMAN_U in t or STUQMAN_L in t:
        return "Attestation"

    # --- DEFENSE ---
    if ATESTAVORMAN in t:
        return "Defense"
    if "master" in t_lower and ("thesis" in t_lower or "defense" in t_lower):
        return "Defense"
    if "thesis is evaluated" in t_lower or "thesis" in t_lower and "novelty" in t_lower:
        return "Defense"

    return ""

## Page parser

In [ ]:
def parse_page(url, log):
    """
    Fetch one YSU program page and return a dict:
      {(chair_code_str, course_name_normalized): {semester, description, assessment}}
    """
    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)
        resp.raise_for_status()
    except requests.RequestException as e:
        log.append(f"  ERROR fetching {url}: {e}")
        return {}

    soup = BeautifulSoup(resp.text, "html.parser")

    main_rows    = soup.find_all("tr", class_="main-body-content")
    content_rows = soup.find_all("tr", class_="content-body-description")

    if len(main_rows) != len(content_rows):
        log.append(f"  WARNING: row count mismatch on {url} "
                   f"({len(main_rows)} main vs {len(content_rows)} content)")

    details = {}
    pairs = min(len(main_rows), len(content_rows))

    for i in range(pairs):
        main = main_rows[i]
        content = content_rows[i]

        tds = main.find_all("td")
        if len(tds) < 2:
            continue
        chair_code  = normalize_text(tds[0].get_text(strip=True))
        course_name = normalize_text(tds[1].get_text(strip=True))
        if not course_name or course_name == "Name of the course":
            continue

        article = content.find("article")
        if not article:
            continue

        # Semester
        sem_div = article.find("div", class_=lambda c: c and "field--name-field-subject-semester" in c)
        semester = None
        if sem_div:
            sem_text = normalize_text(sem_div.get_text(strip=True))
            semester = extract_semester_number(sem_text)

        # Description: prefer section 3, fall back to section 1 (Purpose)
        desc = get_field_text(article, "field--name-field-subject-description")
        if not desc:
            desc = get_field_text(article, "field--name-field-subject-purpose")

        # Assessment
        criteria = get_field_text(article, "field--name-field-subject-criteria")
        assessment = classify_assessment(criteria)

        key = (chair_code, course_name)
        details[key] = {
            "semester":   semester,
            "description": desc,
            "assessment": assessment,
        }

    log.append(f"  Parsed {pairs} course panels from {url}")
    return details

## Run enrichment

In [ ]:
log = ["=== YSU ENRICHMENT LOG ===", ""]

# Load final dataset
df = pd.read_csv(FINAL)
log.append(f"Loaded {len(df)} rows from final_curriculum_dataset.csv")

# Work only on YSU rows
ysu_mask = df["university"] == "Yerevan State University"
log.append(f"YSU rows: {ysu_mask.sum()}")

# Get unique source URLs
ysu_urls = df.loc[ysu_mask, "source_url"].dropna().unique().tolist()
ysu_urls = [normalize_url(u) for u in ysu_urls]
log.append(f"Unique YSU program URLs: {len(ysu_urls)}")
log.append("")

# Fetch and parse all pages
all_details = {}   # (normalized_url, chair_code, course_name) -> detail
for url in ysu_urls:
    log.append(f"Fetching: {url}")
    page_details = parse_page(url, log)
    for (chair_code, course_name), detail in page_details.items():
        all_details[(url, chair_code, course_name)] = detail
    time.sleep(DELAY_S)

log.append("")
log.append(f"Total course detail records fetched: {len(all_details)}")
log.append("")

# Ensure columns exist (they may already be blank for YSU)
if "semester" not in df.columns:
    df["semester"] = ""
if "description" not in df.columns:
    df["description"] = ""
if "assessment" not in df.columns:
    df["assessment"] = ""

matched     = 0
no_detail   = 0
sem_filled  = 0
desc_filled = 0
asm_filled  = 0

for idx, row in df.iterrows():
    if row["university"] != "Yerevan State University":
        continue

    norm_url    = normalize_url(str(row.get("source_url", "")))
    chair_code  = normalize_text(str(row.get("course_code", "")))
    course_name = normalize_text(str(row.get("course_name", "")))

    # Primary key: url + chair_code + name
    key = (norm_url, chair_code, course_name)
    detail = all_details.get(key)

    # Fallback: url + name only (in case chair_code differs by whitespace)
    if detail is None:
        for (u, cc, cn), d in all_details.items():
            if u == norm_url and cn == course_name:
                detail = d
                break

    if detail is None:
        no_detail += 1
        continue

    matched += 1

    # Fill semester (only if currently blank)
    if pd.isna(row.get("semester")) or str(row.get("semester", "")).strip() == "":
        if detail["semester"] is not None:
            df.at[idx, "semester"] = detail["semester"]
            sem_filled += 1

    # Fill description (only if currently blank)
    if pd.isna(row.get("description")) or str(row.get("description", "")).strip() == "":
        if detail["description"]:
            df.at[idx, "description"] = detail["description"]
            desc_filled += 1

    # Fill assessment (only if currently blank)
    if pd.isna(row.get("assessment")) or str(row.get("assessment", "")).strip() == "":
        asm_val = detail["assessment"]
        # Name-based fallbacks when criteria text has no classifiable keyword
        if not asm_val:
            cname = str(row.get("course_name", ""))
            cname_lower = cname.lower()
            if "\u0544\u561\u563\u56B\u057D\u057F\u580\u578\u057D\u561\u056f\u561\u576" in cname or "thesis" in cname_lower or "master" in cname_lower:
                asm_val = "Defense"
            elif "\u0533\u56B\u057F\u561\u056f\u561\u576 \u057D\u565\u574\u56B\u576\u561\u580" in cname or "scientific seminar" in cname_lower:
                asm_val = "Attestation"
            elif "\u0544\u561\u057D\u576\u561\u563\u56B\u057F\u561\u056f\u561\u576 \u057A\u580\u561\u056f" in cname or "professional internship" in cname_lower or "professional practice" in cname_lower:
                asm_val = "Attestation"
            elif "\u0565\u0580\u0565\u0576" in cname:
                asm_val = "Attestation"
        if asm_val:
            df.at[idx, "assessment"] = asm_val
            asm_filled += 1

log.append(f"YSU rows matched to page data:  {matched}")
log.append(f"YSU rows with no page match:    {no_detail}")
log.append(f"  semester filled:    {sem_filled}")
log.append(f"  description filled: {desc_filled}")
log.append(f"  assessment filled:  {asm_filled}")
log.append("")

# Sanity check: row count must not change
assert len(df) == 1161, f"Row count changed! {len(df)}"

# Write back
df.to_csv(FINAL, index=False, encoding="utf-8")
log.append(f"Written: {FINAL}")

# Coverage summary
ysu_df = df[ysu_mask]
log.append("")
log.append("--- YSU FIELD COVERAGE AFTER ENRICHMENT ---")
for col in ["semester", "description", "assessment"]:
    filled = ysu_df[col].replace("", pd.NA).notna().sum()
    pct    = filled / len(ysu_df) * 100
    log.append(f"  {col:15s}: {filled:4d} / {len(ysu_df)} ({pct:.1f}%)")

log_text = "\n".join(log)
print(log_text)

log_path = PROC_DIR / "ysu_enrichment_log.txt"
with open(log_path, "w", encoding="utf-8") as f:
    f.write(log_text + "\n")
print(f"\nLog saved: {log_path}")